<a href="https://colab.research.google.com/github/cewgs/Computer-Vision-Project/blob/main/gradio_interface_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Interface with Gradio
Simply run the cells and a temporary link for the web interface is created.

In [ ]:
!pip install gradio huggingface_hub
import torch
import torch.nn as nn
from torchvision import models, transforms
from huggingface_hub import hf_hub_download
import gradio as gr

In [ ]:
LABEL_MAP = {
    0: "surprise", 1: "fear", 2: "disgust",
    3: "happiness", 4: "sadness", 5: "anger", 6: "neutral"
}

SENTIMENT_MAP = {
    0: "positive", 1: "negative", 2: "negative",
    3: "positive", 4: "negative", 5: "negative", 6: "neutral"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = hf_hub_download(repo_id="cws18/CV_uni_project", filename="emotion_model.pth")

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 7)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def predict(image):
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1).squeeze()
    scores = {LABEL_MAP[i]: float(probs[i]) for i in range(7)}
    top_idx = probs.argmax().item()
    sentiment = SENTIMENT_MAP[top_idx]
    label = f"{LABEL_MAP[top_idx].upper()} | Sentiment: {sentiment}"
    return label, scores

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Text(label="Prediction"),
        gr.Label(label="Emotion probabilities", num_top_classes=7)
    ],
    title="Facial Emotion Recognizer",
    description="Upload a photo of a face to detect the emotion."
)

demo.launch()